In [0]:
%run ./00_config

In [0]:
#creating parameter and paths
import json, re
from datetime import datetime, timezone
from pyspark.sql import functions as F

dbutils.widgets.text("file_name", "ae_2026_03.csv")
file_name = dbutils.widgets.get("file_name").strip()
src   = f"{LANDING}/{file_name}"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
result = None
print(src, stamp)

In [0]:
#Quarantine helper function. Every failure goes through this function
#so quarantined files always arrive with a reason.json explaining why
def quarantine(reason):
    dest_dir = f"{QUARANTINE}/{stamp}"
    dbutils.fs.cp(src, f"{dest_dir}/{file_name}")
    dbutils.fs.put(f"{dest_dir}/reason.json",
                   json.dumps({"file": file_name, "reason": reason, "at": stamp}),
                   overwrite=True)
    dbutils.fs.rm(src)
    return {"status": "quarantined", "file": file_name, "reason": reason}

In [0]:
# 0. Does the file exist in landing?
try:
    dbutils.fs.ls(src)
except Exception:
    result = {"status": "not_found", "file": file_name, "reason": "file_not_in_landing"}

In [0]:
#the various checks, in a sequential order
# 1. File name
m = re.match(FILE_PATTERN, file_name)
if result is None and not m:

    result = quarantine("bad_file_name")

# 2. Structure
if result is None:
    year, month = m.group(1), m.group(2)
    expected_period = f"{year}-{month}-01"
    df = spark.read.format("csv").option("header", True).load(src)
    missing    = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    unexpected = [c for c in df.columns if c not in EXPECTED_COLUMNS]
    if missing or unexpected:
        result = quarantine(f"schema_mismatch missing={missing} unexpected={unexpected}")
    elif df.count() == 0:
        result = quarantine("empty_file")

# 3. Period inside the file must match the file name
if result is None:
    found = [str(r.p) for r in (
        df.filter(F.col("Org Code") != "TOTAL")
          .select(F.expr("try_to_date(initcap(regexp_extract(Period, '([A-Za-z]+-[0-9]{4})$', 1)), 'MMMM-yyyy')").alias("p"))
          .distinct().collect())]
    if found != [expected_period]:
        result = quarantine(f"period_mismatch expected={expected_period} found={found}")

print(result or "All checks passed")

In [0]:
#Now move to bronze
#the bronze is never edited. Files are organized into year/month folders,with 
#timestamps. Hence a revised file sits next to the original instead of overwriting it
if result is None:
    dest = f"{BRONZE}/{year}/{month}/{stamp}_{file_name}"
    dbutils.fs.cp(src, dest)
    dbutils.fs.rm(src)
    result = {"status": "bronze", "file": file_name,
              "bronze_path": dest, "period": expected_period}
print(result)

In [0]:
#run the log...an audit trail of every outcome
spark.sql("""CREATE TABLE IF NOT EXISTS nhsae.ops.run_log (
  run_at TIMESTAMP, stage STRING, file_name STRING, status STRING, detail STRING)""")

(spark.createDataFrame([(file_name, result["status"], json.dumps(result))],
                       "file_name string, status string, detail string")
   .select(F.current_timestamp().alias("run_at"), F.lit("landing_to_bronze").alias("stage"),
           "file_name", "status", "detail")
   .write.mode("append").saveAsTable("nhsae.ops.run_log"))

In [0]:
dbutils.jobs.taskValues.set(key="status", value=result["status"])
dbutils.jobs.taskValues.set(key="bronze_path", value=result.get("bronze_path", ""))

In [0]:
#Hand the result to ADF, and ADF reads this output, and the pipeline later
#uses this bronze_path to tell 03_bronze_to_silver which file to process
dbutils.notebook.exit(json.dumps(result))